In [38]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [40]:
load_dotenv()  # reads .env from current folder
key = os.getenv("OPENAI_API_KEY")
if not key:
    raise ValueError("OPENAI_API_KEY not found in .env")

In [19]:
client = OpenAI(api_key=key)

In [20]:

def process_ticket(ticket_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Extract issue_type and urgency_level (1-5) as JSON."},
            {"role": "user", "content": ticket_text}
            ]
            )
    return json.loads(response.choices[0].message.content)

In [27]:
tk1 = "My laptop won’t turn on after yesterday’s update. I have a client demo in one hour and I can’t access any files."
tk2 = "The printer on floor 3 is showing a paper jam error, but there’s no paper stuck. Please check when possible."
tk3 = "I forgot my VPN password and can’t log in from home. Not urgent, but I need it reset before tomorrow morning."
tk4 = "Production website is down and customers are reporting checkout failures. This is impacting sales right now."
tk5 = "Can someone install Adobe Photoshop on my workstation this week? I need it for a new design project."
tk6 = "Email is delayed by about 10 minutes for all users in the finance team. No outages yet, but it’s causing confusion."
tk7 = "There is a suspicious login alert on my account from another country. Please investigate immediately."
tk8 = "The office Wi-Fi in meeting room B keeps disconnecting during calls. It happens several times a day."
tk9 = "Requesting access to the analytics dashboard for the marketing intern starting next Monday."
tk10 = "Database backup job failed overnight for the second day in a row. We need this fixed before the next backup window."

tks= [tk1, tk2, tk3, tk4, tk5, tk6, tk7, tk8, tk9, tk10]


In [30]:
results = []

for ticket in tks:
    parsed = process_ticket(ticket)
    results.append(
        {
            "ticket": ticket,
            "issue_type": parsed.get("issue_type"),
            "urgency_level": parsed.get("urgency_level"),
        }
    )

df = pd.DataFrame(results, columns=["ticket", "issue_type", "urgency_level"])
df


,ticket,issue_type,urgency_level
0,My laptop won’t turn on after yesterday’s upda...,Laptop won't turn on after update,5
1,The printer on floor 3 is showing a paper jam ...,printer error,3
2,I forgot my VPN password and can’t log in from...,VPN password reset,3
3,Production website is down and customers are r...,Website Down,1
4,Can someone install Adobe Photoshop on my work...,Software Installation,3
5,Email is delayed by about 10 minutes for all u...,Email delay,3
6,There is a suspicious login alert on my accoun...,suspicious login alert,5
7,The office Wi-Fi in meeting room B keeps disco...,Wi-Fi connectivity issue,3
8,Requesting access to the analytics dashboard f...,Access Request,3
9,Database backup job failed overnight for the s...,Database backup job failure,4


In [31]:
df.to_csv("./tickets.csv", index=False)